# Learned Stage-Wise Curvature Tuning on DTD

This notebook evaluates Learned Stage-Wise Curvature Tuning (SW-CT) on the Describable Textures Dataset (DTD) using an ImageNet-pretrained ResNet-18.

DTD is a texture-recognition dataset, making it different from the Beans dataset used in the previous experiment. Testing on DTD helps determine whether the behavior of Learned SW-CT generalizes to a different visual classification problem.

The experiments compare four conditions:

1. **Baseline** — the original pretrained ResNet with ReLU activations and a trained downstream classifier.
2. **Single-Parameter Curvature Tuning (S-CT)** — one global beta selected through validation-based exhaustive search.
3. **Learned SW-CT with fixed initialization** — four trainable stage-wise beta parameters initialized at 0.78.
4. **Learned SW-CT with random initialization** — four trainable stage-wise beta parameters independently initialized between 0.70 and 0.99.

Each condition is evaluated using seeds 42, 43, and 44.

Experiments are conducted for both 20 and 30 epochs to examine whether additional optimization improves accuracy or changes the learned stage-wise beta values.

Wall-clock execution time is recorded to compare the computational cost of Baseline, S-CT, and Learned SW-CT.

In [1]:
!nvidia-smi

Thu Oct  1 02:23:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%cd /content

!git clone -b stage-wise-ct \
    https://github.com/Syed1611/curvature-tuning-research.git

%cd /content/curvature-tuning-research/src/curvature-tuning

!git branch --show-current
!git log -1 --oneline

/content
Cloning into 'curvature-tuning-research'...
remote: Enumerating objects: 563, done.
remote: Counting objects: 100% (235/235), done.
remote: Compressing objects: 100% (112/112), done.
remote: Total 563 (delta 147), reused 179 (delta 102), pack-reused 328 (from 1)
Receiving objects: 100% (563/563), 6.52 MiB | 7.06 MiB/s, done.
Resolving deltas: 100% (278/278), done.
/content/curvature-tuning-research/src/curvature-tuning
stage-wise-ct
c0c969f (HEAD -> stage-wise-ct, origin/stage-wise-ct, origin/HEAD) updated OCTMNIST notebook


## Environment Setup

The repository dependencies are installed before running the experiments.

After installation, the Colab runtime should be restarted so that the required package versions are loaded correctly.

In [3]:
%pip install -q -r requirements.txt
%pip install --force-reinstall --no-cache-dir "numpy==1.26.3"

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 8.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 8.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 45.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.2/133.2 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 54.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 125.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 144.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 101.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13

In [1]:
%cd /content/curvature-tuning-research/src/curvature-tuning

import os
import torch

os.environ["WANDB_MODE"] = "disabled"

print("Working directory:", os.getcwd())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/content/curvature-tuning-research/src/curvature-tuning
Working directory: /content/curvature-tuning-research/src/curvature-tuning
PyTorch: 2.6.0+cu124
CUDA available: True
GPU: Tesla T4


## DTD Dataset

Before running the experiments, the DTD data loaders are checked to confirm that the downstream dataset can be loaded correctly.

The same dataset split and random seed protocol are used for Baseline, S-CT, and Learned SW-CT so that the methods are compared under the same conditions.

In [2]:
from utils.data import get_data_loaders

train_loader, test_loader, val_loader = get_data_loaders(
    "imagenet_to_dtd",
    train_batch_size=32,
    test_batch_size=800,
    seed=42,
)

print("Train samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels:", labels[:10])

100%|██████████| 625M/625M [00:42<00:00, 14.8MB/s] 


Train samples: 1880
Validation samples: 1880
Test samples: 1880
Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Labels: tensor([12, 28, 12, 14, 24, 10,  5, 39, 17, 17])


# ResNet-18 Experiments

ResNet-18 is evaluated first because it was used during the initial development of Learned SW-CT.

The experiment compares Baseline, S-CT, fixed-initialization SW-CT, and random-initialization SW-CT.

All methods use the same ImageNet-pretrained backbone, DTD dataset split, experimental seeds, and downstream training protocol.

## ResNet-18 — 20-Epoch Evaluation

The first evaluation uses 20 training epochs.

This provides the primary reference experiment and maintains compatibility with the earlier Learned SW-CT work.

### Methods

**Baseline**

The pretrained ResNet-18 retains its original ReLU activations. The backbone is frozen and only the downstream classifier is trained.

**S-CT**

The original Single-Parameter Curvature Tuning method searches over a single beta shared throughout the network. Candidate beta values from 0.70 to 1.00 are evaluated using validation performance, and the best candidate is evaluated on the test set.

The recorded S-CT execution time includes the complete beta search rather than only the final selected beta.

**Learned SW-CT — Fixed Initialization**

Four trainable beta parameters are assigned to the four ResNet stages. All four begin at beta = 0.78.

This preserves the initialization used in the original Learned SW-CT experiments.

**Learned SW-CT — Random Initialization**

Four trainable stage-wise beta parameters are initialized independently between 0.70 and 0.99.

The random initialization is generated deterministically from the experiment seed. This allows the experiment to test whether Learned SW-CT depends on the fixed 0.78 starting value while remaining reproducible.

In [3]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-01 02:28:20.804 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_dtd_resnet18_seed42_epochs20.log
2026-10-01 02:28:20.811 | INFO     | __main__:main:78 - Running on cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 231MB/s]
2026-10-01 02:28:21.792 | INFO     | __main__:main:105 - Testing baseline...
2026-10-01 02:28:21.812 | INFO     | __main__:main:116 - Number of trainable parameters: 24111
2026-10-01 02:28:21.813 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-01 02:28:44.128 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.976914, Accuracy: 0.00%
2026-10-01 02:28:44.421 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 3.025137, Val Accuracy: 32.07%
2026-10-01 02:28:44.422 | INFO     | train:linear_probe:190 - New best validation accuracy: 32.07 at epoch 1
2026-10-01 02:28:44.467 | INFO   

In [4]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-01 02:44:44.236 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_dtd_resnet18_seed43_epochs20.log
2026-10-01 02:44:44.239 | INFO     | __main__:main:78 - Running on cuda
2026-10-01 02:44:44.993 | INFO     | __main__:main:105 - Testing baseline...
2026-10-01 02:44:45.013 | INFO     | __main__:main:116 - Number of trainable parameters: 24111
2026-10-01 02:44:45.013 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-01 02:45:06.054 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.055932, Accuracy: 0.00%
2026-10-01 02:45:06.379 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 2.993484, Val Accuracy: 30.59%
2026-10-01 02:45:06.380 | INFO     | train:linear_probe:190 - New best validation accuracy: 30.59 at epoch 1
2026-10-01 02:45:06.434 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 2.764283, Accuracy: 31.25%
2026-10-01 02:45:06.743 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 2.053752, Val Accu

In [5]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-01 03:01:10.312 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_dtd_resnet18_seed44_epochs20.log
2026-10-01 03:01:10.316 | INFO     | __main__:main:78 - Running on cuda
2026-10-01 03:01:11.022 | INFO     | __main__:main:105 - Testing baseline...
2026-10-01 03:01:11.042 | INFO     | __main__:main:116 - Number of trainable parameters: 24111
2026-10-01 03:01:11.042 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-01 03:01:33.067 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.173090, Accuracy: 0.00%
2026-10-01 03:01:33.387 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 3.035984, Val Accuracy: 31.22%
2026-10-01 03:01:33.387 | INFO     | train:linear_probe:190 - New best validation accuracy: 31.22 at epoch 1
2026-10-01 03:01:33.447 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 2.947498, Accuracy: 37.50%
2026-10-01 03:01:33.767 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 2.087505, Val Accu

### Learned SW-CT with Fixed Beta Initialization

The original Learned SW-CT configuration is first reproduced using the fixed initialization:

\[[\beta_1,\beta_2,\beta_3,\beta_4]=[0.78,0.78,0.78,0.78].\]

The initial value is only a starting point. During training, the four beta parameters are optimized independently together with the downstream classifier.

These runs provide the reference against which random initialization will be compared.

In [6]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-01 03:17:45.845 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_dtd_resnet18_seed42_epochs20_initfixed0.78_betalr0.01.log
2026-10-01 03:17:45.904 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 03:17:46.368 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 03:17:46.368 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 03:17:46.394 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 03:17:46.395 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 03:17:46.395 | INFO     | __main__:main:439 - Total trainable parameters: 24115
2026-10-01 03:17:47.825 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.106285, Accuracy: 9.38%
2026-10-01 03:18:11.364 | INFO     | train:test_epoch:90 - Epoch

In [7]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-01 03:26:10.687 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_dtd_resnet18_seed43_epochs20_initfixed0.78_betalr0.01.log
2026-10-01 03:26:10.748 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 03:26:11.212 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 03:26:11.212 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 03:26:11.234 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 03:26:11.235 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 03:26:11.235 | INFO     | __main__:main:439 - Total trainable parameters: 24115
2026-10-01 03:26:12.492 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.396770, Accuracy: 0.00%
2026-10-01 03:26:36.006 | INFO     | train:test_epoch:90 - Epoch

In [8]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-01 03:34:35.612 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_dtd_resnet18_seed44_epochs20_initfixed0.78_betalr0.01.log
2026-10-01 03:34:35.673 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 03:34:36.129 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 03:34:36.129 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 03:34:36.157 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 03:34:36.157 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 03:34:36.157 | INFO     | __main__:main:439 - Total trainable parameters: 24115
2026-10-01 03:34:37.325 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.886103, Accuracy: 9.38%
2026-10-01 03:35:01.319 | INFO     | train:test_epoch:90 - Epoch

### Learned SW-CT with Random Beta Initialization

The fixed beta = 0.78 initialization may raise the question of whether Learned SW-CT performs well only because its starting point was informed by the earlier S-CT experiments.

To test this dependency, each of the four stage-wise beta parameters is now initialized independently from a uniform distribution between 0.70 and 0.99.

The initialization is reproducible: the random generator is seeded using the experiment seed. Therefore, each seed always receives the same initial beta vector.

If random initialization produces similar performance to the fixed 0.78 experiment, this provides evidence that the Learned SW-CT result is not dependent on a single hand-selected initialization.

In [9]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-01 03:43:08.965 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_dtd_resnet18_seed42_epochs20_initrandom_betalr0.01.log
2026-10-01 03:43:09.041 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 03:43:09.685 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-01 03:43:09.686 | INFO     | __main__:main:394 - Requested initial stage betas: [0.924447, 0.827275, 0.948993, 0.902237]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 03:43:09.722 | INFO     | __main__:main:418 - Initial stage betas: [0.924447238445282, 0.827274739742279, 0.9489933848381042, 0.9022367000579834]
2026-10-01 03:43:09.722 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 03:43:09.723 | INFO     | __main__:main:439 - Total trainable parameters: 24115
2026-10-01 03:43:11.806 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.078638, Accuracy: 9.38%
2026-10-01 03:43:34.530 | INFO     | train:test_epoc

In [10]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-01 03:51:36.552 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_dtd_resnet18_seed43_epochs20_initrandom_betalr0.01.log
2026-10-01 03:51:36.655 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 03:51:37.290 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-01 03:51:37.290 | INFO     | __main__:main:394 - Requested initial stage betas: [0.889167, 0.712695, 0.705809, 0.943372]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 03:51:37.325 | INFO     | __main__:main:418 - Initial stage betas: [0.8891667127609253, 0.7126947641372681, 0.7058085799217224, 0.9433716535568237]
2026-10-01 03:51:37.326 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 03:51:37.326 | INFO     | __main__:main:439 - Total trainable parameters: 24115
2026-10-01 03:51:39.007 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.249947, Accuracy: 0.00%
2026-10-01 03:52:01.257 | INFO     | train:test_ep

In [11]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-01 03:59:57.257 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_dtd_resnet18_seed44_epochs20_initrandom_betalr0.01.log
2026-10-01 03:59:57.315 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 03:59:57.793 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-01 03:59:57.793 | INFO     | __main__:main:394 - Requested initial stage betas: [0.735544, 0.774853, 0.817674, 0.981063]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 03:59:57.816 | INFO     | __main__:main:418 - Initial stage betas: [0.7355440258979797, 0.7748528122901917, 0.8176735043525696, 0.9810633659362793]
2026-10-01 03:59:57.816 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 03:59:57.816 | INFO     | __main__:main:439 - Total trainable parameters: 24115
2026-10-01 03:59:59.065 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.915092, Accuracy: 6.25%
2026-10-01 04:00:22.447 | INFO     | train:test_ep

# ResNet-18 — 30-Epoch Evaluation

The same experiments are now repeated for 30 epochs.

The purpose of increasing the training budget is to determine whether the 20-epoch results are limited by insufficient optimization.

If a method improves substantially with 30 epochs, this suggests that additional optimization is useful. If performance remains similar, it indicates that the method has largely stabilized within the original 20-epoch training budget.

All other experimental settings remain unchanged.

For random-initialization SW-CT, the same seed produces exactly the same initial four-beta vector in both the 20-epoch and 30-epoch experiments. Therefore, initialization is held constant while the training duration changes.

Execution time is again recorded to measure the computational cost of extending training from 20 to 30 epochs.

In [ ]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-01 04:08:15.838 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_dtd_resnet18_seed42_epochs30.log
2026-10-01 04:08:15.841 | INFO     | __main__:main:78 - Running on cuda
2026-10-01 04:08:16.363 | INFO     | __main__:main:105 - Testing baseline...
2026-10-01 04:08:16.376 | INFO     | __main__:main:116 - Number of trainable parameters: 24111
2026-10-01 04:08:16.377 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-01 04:08:37.033 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.976914, Accuracy: 0.00%
2026-10-01 04:08:37.339 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 3.025137, Val Accuracy: 32.07%
2026-10-01 04:08:37.340 | INFO     | train:linear_probe:190 - New best validation accuracy: 32.07 at epoch 1
2026-10-01 04:08:37.396 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 2.898652, Accuracy: 43.75%
2026-10-01 04:08:37.706 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 2.073287, Val Accu

In [23]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-01 05:46:02.478 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_dtd_resnet18_seed43_epochs30.log
2026-10-01 05:46:02.482 | INFO     | __main__:main:78 - Running on cuda
2026-10-01 05:46:03.288 | INFO     | __main__:main:105 - Testing baseline...
2026-10-01 05:46:03.314 | INFO     | __main__:main:116 - Number of trainable parameters: 24111
2026-10-01 05:46:03.315 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-01 05:46:28.582 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.055932, Accuracy: 0.00%
2026-10-01 05:46:29.069 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 2.993484, Val Accuracy: 30.59%
2026-10-01 05:46:29.071 | INFO     | train:linear_probe:190 - New best validation accuracy: 30.59 at epoch 1
2026-10-01 05:46:29.158 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 2.764283, Accuracy: 31.25%
2026-10-01 05:46:29.703 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 2.053752, Val Accu

In [24]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-01 06:08:40.281 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_dtd_resnet18_seed44_epochs30.log
2026-10-01 06:08:40.284 | INFO     | __main__:main:78 - Running on cuda
2026-10-01 06:08:41.045 | INFO     | __main__:main:105 - Testing baseline...
2026-10-01 06:08:41.071 | INFO     | __main__:main:116 - Number of trainable parameters: 24111
2026-10-01 06:08:41.072 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-01 06:09:02.679 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.173090, Accuracy: 0.00%
2026-10-01 06:09:03.108 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 3.035984, Val Accuracy: 31.22%
2026-10-01 06:09:03.110 | INFO     | train:linear_probe:190 - New best validation accuracy: 31.22 at epoch 1
2026-10-01 06:09:03.186 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 2.947497, Accuracy: 37.50%
2026-10-01 06:09:03.607 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 2.087505, Val Accu

In [25]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 06:31:21.903 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_dtd_resnet18_seed42_epochs30_initfixed0.78_betalr0.01.log
2026-10-01 06:31:22.012 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 06:31:22.802 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 06:31:22.803 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 06:31:22.849 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 06:31:22.850 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 06:31:22.850 | INFO     | __main__:main:439 - Total trainable parameters: 24115
2026-10-01 06:31:24.663 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.106285, Accuracy: 9.38%
2026-10-01 06:31:50.586 | INFO     | train:test_epoch:90 - Epoch

In [26]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 06:45:34.508 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_dtd_resnet18_seed43_epochs30_initfixed0.78_betalr0.01.log
2026-10-01 06:45:34.588 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 06:45:35.297 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 06:45:35.297 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 06:45:35.340 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 06:45:35.341 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 06:45:35.342 | INFO     | __main__:main:439 - Total trainable parameters: 24115
2026-10-01 06:45:36.844 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.396769, Accuracy: 0.00%
2026-10-01 06:46:01.005 | INFO     | train:test_epoch:90 - Epoch

In [27]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 06:58:55.332 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_dtd_resnet18_seed44_epochs30_initfixed0.78_betalr0.01.log
2026-10-01 06:58:55.395 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 06:58:55.917 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 06:58:55.918 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 06:58:55.947 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 06:58:55.948 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 06:58:55.948 | INFO     | __main__:main:439 - Total trainable parameters: 24115
2026-10-01 06:58:57.305 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.886103, Accuracy: 9.38%
2026-10-01 06:59:18.328 | INFO     | train:test_epoch:90 - Epoch

In [28]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 07:10:50.650 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_dtd_resnet18_seed42_epochs30_initrandom_betalr0.01.log
2026-10-01 07:10:50.752 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 07:10:51.482 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-01 07:10:51.483 | INFO     | __main__:main:394 - Requested initial stage betas: [0.924447, 0.827275, 0.948993, 0.902237]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 07:10:51.526 | INFO     | __main__:main:418 - Initial stage betas: [0.924447238445282, 0.827274739742279, 0.9489933848381042, 0.9022367000579834]
2026-10-01 07:10:51.527 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 07:10:51.527 | INFO     | __main__:main:439 - Total trainable parameters: 24115
2026-10-01 07:10:53.247 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.078638, Accuracy: 9.38%
2026-10-01 07:11:16.762 | INFO     | train:test_epoc

In [29]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 07:22:47.231 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_dtd_resnet18_seed43_epochs30_initrandom_betalr0.01.log
2026-10-01 07:22:47.314 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 07:22:47.916 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-01 07:22:47.916 | INFO     | __main__:main:394 - Requested initial stage betas: [0.889167, 0.712695, 0.705809, 0.943372]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 07:22:47.949 | INFO     | __main__:main:418 - Initial stage betas: [0.8891667127609253, 0.7126947641372681, 0.7058085799217224, 0.9433716535568237]
2026-10-01 07:22:47.950 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 07:22:47.950 | INFO     | __main__:main:439 - Total trainable parameters: 24115
2026-10-01 07:22:49.237 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.249947, Accuracy: 0.00%
2026-10-01 07:23:10.933 | INFO     | train:test_ep

In [30]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds dtd \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 07:34:39.036 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_dtd_resnet18_seed44_epochs30_initrandom_betalr0.01.log
2026-10-01 07:34:39.108 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 07:34:39.714 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-01 07:34:39.715 | INFO     | __main__:main:394 - Requested initial stage betas: [0.735544, 0.774853, 0.817674, 0.981063]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 07:34:39.763 | INFO     | __main__:main:418 - Initial stage betas: [0.7355440258979797, 0.7748528122901917, 0.8176735043525696, 0.9810633659362793]
2026-10-01 07:34:39.765 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 07:34:39.765 | INFO     | __main__:main:439 - Total trainable parameters: 24115
2026-10-01 07:34:41.080 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.915092, Accuracy: 6.25%
2026-10-01 07:35:00.436 | INFO     | train:test_ep